# W2/D1 Alert Correlation

Goal: reduce 20 raw alerts into a small number of meaningful clusters using session windows, alert fingerprints, and service topology.

In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import networkx as nx
import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'd1':
    BASE_DIR = BASE_DIR / 'w2' / 'd1'

DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ALERTS_PATH = DATA_DIR / 'alerts_sample.jsonl'
SERVICES_PATH = DATA_DIR / 'services.json'
OUTPUT_PATH = RESULTS_DIR / 'cluster_summary.json'

print(f'base_dir={BASE_DIR}')
print(f'alerts_path={ALERTS_PATH}')
print(f'services_path={SERVICES_PATH}')

base_dir=E:\AIO\Project\repo-aiops-minhtq\w2\d1
alerts_path=E:\AIO\Project\repo-aiops-minhtq\w2\d1\data\alerts_sample.jsonl
services_path=E:\AIO\Project\repo-aiops-minhtq\w2\d1\data\services.json


In [2]:
def load_alerts(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def load_services(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


alerts = load_alerts(ALERTS_PATH)
services_doc = load_services(SERVICES_PATH)

alert_df = pd.DataFrame(alerts)
print(f'loaded_alerts={len(alerts)}')
display(alert_df[['id', 'ts', 'service', 'metric', 'severity']].head(20))

loaded_alerts=20


,id,ts,service,metric,severity
0,a-0001,2026-06-12T09:42:01Z,payment-svc,db_connection_pool_used_ratio,warn
1,a-0002,2026-06-12T09:42:18Z,payment-svc,db_connection_pool_used_ratio,crit
2,a-0003,2026-06-12T09:42:22Z,payment-svc,latency_p99_ms,crit
3,a-0004,2026-06-12T09:42:30Z,payment-svc,error_rate,warn
4,a-0005,2026-06-12T09:42:45Z,checkout-svc,latency_p99_ms,warn
5,a-0006,2026-06-12T09:43:01Z,checkout-svc,downstream_payment_error_rate,crit
6,a-0007,2026-06-12T09:43:15Z,edge-lb,upstream_5xx_rate,warn
7,a-0008,2026-06-12T09:43:18Z,payment-svc,latency_p99_ms,crit
8,a-0009,2026-06-12T09:43:32Z,cart-svc,latency_p99_ms,warn
9,a-0010,2026-06-12T09:43:50Z,notification-svc,queue_lag_ms,warn


In [3]:
def parse_ts(value: str) -> datetime:
    return datetime.fromisoformat(value.replace('Z', '+00:00')).astimezone(timezone.utc)


def fingerprint(alert: dict) -> str:
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"


def is_explicit_noise(alert: dict) -> bool:
    note = alert.get('labels', {}).get('note', '').lower()
    return 'unrelated' in note or 'noise' in note or 'independent' in note


def max_severity(alerts: list[dict]) -> str:
    rank = {'info': 0, 'warn': 1, 'crit': 2}
    return max((a['severity'] for a in alerts), key=lambda sev: rank.get(sev, -1))


def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: parse_ts(a['ts']))
    groups = [[sorted_alerts[0]]]
    for alert in sorted_alerts[1:]:
        previous_ts = parse_ts(groups[-1][-1]['ts'])
        if (parse_ts(alert['ts']) - previous_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups


def build_service_graph(services_doc: dict) -> nx.DiGraph:
    graph = nx.DiGraph()
    for item in services_doc.get('services', []):
        graph.add_node(item['name'], kind='service', **item)
    for item in services_doc.get('stores', []):
        graph.add_node(item['name'], kind='store', **item)
    for edge in services_doc.get('edges', []):
        graph.add_edge(edge['from'], edge['to'], **edge)
    return graph


def topology_group(alerts: list[dict], graph: nx.DiGraph, max_hop: int = 2) -> list[list[dict]]:
    undirected = graph.to_undirected()
    normal_alerts = [a for a in alerts if not is_explicit_noise(a)]
    forced_orphans = [[a] for a in alerts if is_explicit_noise(a)]

    by_service: dict[str, list[dict]] = defaultdict(list)
    for alert in normal_alerts:
        by_service[alert['service']].append(alert)

    services = list(by_service)
    parent = {service: service for service in services}

    def find(service: str) -> str:
        while parent[service] != service:
            parent[service] = parent[parent[service]]
            service = parent[service]
        return service

    def union(a: str, b: str) -> None:
        root_a = find(a)
        root_b = find(b)
        if root_a != root_b:
            parent[root_b] = root_a

    for i, service_a in enumerate(services):
        for service_b in services[i + 1:]:
            try:
                distance = nx.shortest_path_length(undirected, service_a, service_b)
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue
            if distance <= max_hop:
                union(service_a, service_b)

    grouped: dict[str, list[dict]] = defaultdict(list)
    for service in services:
        grouped[find(service)].extend(by_service[service])

    return list(grouped.values()) + forced_orphans


def summarize_cluster(cluster_id: str, group: list[dict]) -> dict:
    sorted_group = sorted(group, key=lambda a: parse_ts(a['ts']))
    return {
        'cluster_id': cluster_id,
        'alert_count': len(sorted_group),
        'services': sorted({a['service'] for a in sorted_group}),
        'time_range': [min(a['ts'] for a in sorted_group), max(a['ts'] for a in sorted_group)],
        'max_severity': max_severity(sorted_group),
        'fingerprints': sorted({fingerprint(a) for a in sorted_group}),
        'alert_ids': [a['id'] for a in sorted_group],
    }


def correlate(alerts: list[dict], graph: nx.DiGraph, gap_sec: int = 120, max_hop: int = 2) -> dict:
    clusters = []
    for session_idx, session_alerts in enumerate(session_groups(alerts, gap_sec=gap_sec), start=1):
        for group_idx, group in enumerate(topology_group(session_alerts, graph, max_hop=max_hop)):
            clusters.append(summarize_cluster(f'c-{session_idx:03d}-{group_idx:03d}', group))

    return {
        'input_alerts': len(alerts),
        'output_clusters': len(clusters),
        'reduction_ratio': round(1 - len(clusters) / len(alerts), 4) if alerts else 0,
        'params': {'gap_sec': gap_sec, 'max_hop': max_hop},
        'clusters': clusters,
    }


graph = build_service_graph(services_doc)
print(f'nodes={graph.number_of_nodes()}, edges={graph.number_of_edges()}')
print('functions_ready=True')

nodes=14, edges=17
functions_ready=True


In [4]:
summary = correlate(alerts, graph, gap_sec=120, max_hop=2)

cluster_rows = []
for cluster in summary['clusters']:
    cluster_rows.append({
        'cluster_id': cluster['cluster_id'],
        'alert_count': cluster['alert_count'],
        'services': ', '.join(cluster['services']),
        'time_range': ' -> '.join(cluster['time_range']),
        'max_severity': cluster['max_severity'],
        'alert_ids': ', '.join(cluster['alert_ids']),
    })

print(f"input_alerts={summary['input_alerts']}")
print(f"output_clusters={summary['output_clusters']}")
print(f"reduction_ratio={summary['reduction_ratio']}")
display(pd.DataFrame(cluster_rows))

input_alerts=20
output_clusters=3
reduction_ratio=0.85


,cluster_id,alert_count,services,time_range,max_severity,alert_ids
0,c-001-000,18,"cart-svc, checkout-svc, edge-lb, notification-...",2026-06-12T09:42:01Z -> 2026-06-12T09:48:30Z,crit,"a-0001, a-0002, a-0003, a-0004, a-0005, a-0006..."
1,c-001-001,1,recommender-svc,2026-06-12T09:45:10Z -> 2026-06-12T09:45:10Z,warn,a-0013
2,c-001-002,1,search-svc,2026-06-12T09:46:50Z -> 2026-06-12T09:46:50Z,warn,a-0016


In [5]:
with OUTPUT_PATH.open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

with OUTPUT_PATH.open('r', encoding='utf-8') as f:
    loaded_summary = json.load(f)

all_clustered_ids = [alert_id for cluster in loaded_summary['clusters'] for alert_id in cluster['alert_ids']]
assert loaded_summary['input_alerts'] == 20
assert 3 <= loaded_summary['output_clusters'] <= 5
assert loaded_summary['reduction_ratio'] >= 0.5
assert len(set(all_clustered_ids)) == loaded_summary['input_alerts']
assert all(cluster['services'] and cluster['time_range'] for cluster in loaded_summary['clusters'])

print(f'wrote={OUTPUT_PATH}')
print('valid_json=True')
print('all_acceptance_checks=True')

wrote=E:\AIO\Project\repo-aiops-minhtq\w2\d1\results\cluster_summary.json
valid_json=True
all_acceptance_checks=True
